In [1]:
import asyncio

import logging
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter1d
from sklearn.metrics.pairwise import cosine_similarity

from matplotlib.collections import PathCollection
from matplotlib.legend_handler import HandlerPathCollection, HandlerLine2D

from mime_db import MimeDb

For a DTW timecode match file:

Need to get DB IDs for both videos
Also get framerates for each video

Lines are always
ref_stride comp_stride ref_time comp_time
1 1 0.18575963718820862 0.18575963718820862\*

- = both the reference time and comparison time have advanced

For each line in the match file
If it doesn't end in \*, skip it (this means it involves an asynchrony)

Get the poses for the frame closest to the timecode in each video


In [2]:
db = await MimeDb.create()

video_meta = await db.get_available_videos()
videos_df = pd.DataFrame.from_records(video_meta, columns=video_meta[0].keys())

In [3]:
from scipy.stats import skew, skewtest, kurtosis, kurtosistest

def get_distribution_stats(distrib, plot=False):

    if len(distrib) == 0:
        return {"count": 0, "mean": 0, "median": 0, "stdev": 0, "skewness": 0, "kurtosis": 0}

    if skewtest(distrib).pvalue < .05:
        skewness = skew(distrib)
    else:
        skewness = 0

    if kurtosistest(distrib).pvalue < .05:
        kurtosis_value = kurtosis(distrib)
    else:
        kurtosis_value = 0

    if plot:
        plt.hist(distrib, bins="auto")  # arguments are passed to np.histogram
        plt.show()

    return {"count": len(distrib), "mean": np.mean(distrib), "median": np.median(distrib), "stdev": np.std(distrib), "skewness": skewness, "kurtosis": kurtosis_value}

In [ ]:
def get_id_and_fps(video_name, videos_df):
    video_df = videos_df[videos_df["video_name"] == video_name]
    return video_df["id"].values[0], video_df["fps"].values[0]

reference_video = "Don_Giovanni-Hampe-Karajan_jFPqTCR0_F8.mp4"

reference_id, reference_fps = get_id_and_fps(reference_video, videos_df)

print("Getting reference poses from DB")
reference_poses = await db.get_pose_data_from_video(reference_id)
ref_poses_df = pd.DataFrame.from_records(reference_poses, columns=reference_poses[0].keys())
reference_frames = await db.get_pose_data_by_frame(reference_id)
ref_frames_df = pd.DataFrame.from_records(reference_frames, columns=reference_frames[0].keys())

In [ ]:
comparison_videos = ["Don_Giovanni_Mozart.mp4", # Castellucci
"Don_Giovanni-Bechtolf-Zurich_zjRtCS27udc.mp4",
"Don_Giovanni-Flimm-Harnoncourt_aL2VdxseTvE.mp4",
"Don_Giovanni-Michieletto-La-Fenice_E1d2h7tMDmI.mp4",
"Don_Giovanni-Sivadier-Aix_8wEMzWH52FA.mp4",
"Don_Giovanni-Soderblom-Helsinki_UzxYEVbOS5w.mp4"]

all_videos = ["Don_Giovanni-Hampe-Karajan_jFPqTCR0_F8.mp4"] + comparison_videos

video_aliases = {
"Don_Giovanni-Hampe-Karajan_jFPqTCR0_F8.mp4": "Hampe",
"Don_Giovanni_Mozart.mp4": "Castellucci",
"Don_Giovanni-Bechtolf-Zurich_zjRtCS27udc.mp4": "Bechtolf",
"Don_Giovanni-Flimm-Harnoncourt_aL2VdxseTvE.mp4": "Flimm", 
"Don_Giovanni-Michieletto-La-Fenice_E1d2h7tMDmI.mp4": "Michieletto",
"Don_Giovanni-Sivadier-Aix_8wEMzWH52FA.mp4": "Sivadier",
"Don_Giovanni-Soderblom-Helsinki_UzxYEVbOS5w.mp4": "Soderblom"
}

# Adapted from https://stackoverflow.com/questions/12498673/plot-axis-of-timedeltas-formatted-as-dd-hhmmss-with-matplotlib
def HMS(seconds, _):
    seconds = int(seconds)
    hours = int(seconds / 3600)
    seconds -= 3600 * hours
    minutes = int(seconds / 60.0)
    seconds -= 60 * minutes
    return "%02d:%02d:%02d" % (hours, minutes, int(seconds))

def smooth_series(vals, window_size = 10):
    return uniform_filter1d(vals, size=window_size)

REF_BOUNDARIES = [65,
427,
1260,
2105,
3708,
4040,
4227,
5189,
5756,
5911,
7243,
8650,
9177,
9750,
10267,
10749,
11122]

REF_ACTS = [427, 5911]

video_aliases_legend = [video_aliases[video_name] for video_name in all_videos]

video_colors = ["orange", "cyan", "blue", "green", "purple", "gray", "red"]

def update_legend(handle, orig):
    handle.update_from(orig)
    handle.set_alpha(1)

all_movement3d_fig = plt.figure(figsize=(16,6))
ax = all_movement3d_fig.gca()
ax.set_title("3D movement in each performance")
ax.xaxis.set_major_formatter(plt.FuncFormatter(HMS))

# For building the average "consensus" pose at each time point shared by at least two videos
aligned_ref_poses = {}
aligned_comp_poses = {}
aligned_ref_frames = {}
aligned_comp_frames = {}

plotted_ref_movement = False

for c, comparison_video in enumerate(comparison_videos):

    video_color = video_colors[c+1]

    comparison_id, comparison_fps = get_id_and_fps(comparison_video, videos_df)
    print("Getting comparison poses for video", comparison_video)
    comparison_poses = await db.get_pose_data_from_video(comparison_id)
    print("Getting comparison frames for video", comparison_video)
    comparison_frames = await db.get_pose_data_by_frame(comparison_id)

    print("Building dataframes")
    comp_poses_df = pd.DataFrame.from_records(comparison_poses, columns=comparison_poses[0].keys())
    comp_frames_df = pd.DataFrame.from_records(comparison_frames, columns=comparison_frames[0].keys())

    sim_times = []
    poem_sim_values = []
    global3d_sim_values = []
    ava_sim_values = []
    movement3d_sim_values = []

    raw_comp_movement3d = []
    raw_ref_movement3d = []

    print("Matching timecodes and frames by warping path")
    with open(f"stylometry/alignment_files_4096/{comparison_video}_warping.txt", "r", encoding="utf-8") as warping_data:
        for line in warping_data:
            line_data = line.strip().split("\t")
            if line_data[0] == 0 or line_data[3].endswith("*"):
                continue
            reference_seconds = float(line_data[2])
            comparison_seconds = float(line_data[3])
            reference_frame = round(reference_seconds * reference_fps)
            comparison_frame = round(comparison_seconds * comparison_fps)

            matching_ref_poses = ref_poses_df[ref_poses_df["frame"] == reference_frame]
            matching_comp_poses = comp_poses_df[comp_poses_df["frame"] == comparison_frame]

            matching_ref_frames = ref_frames_df[ref_frames_df["frame"] == reference_frame]
            matching_comp_frames = comp_frames_df[comp_frames_df["frame"] == comparison_frame]

            # Note that we exclude frames from both the single-video-to-reference comparison
            # AND the consensus averages if there's no direct match (could still include the
            # non-matching poses in the latter, but it's cleaner if we don't)
            if len(matching_ref_poses) == 0 or len(matching_comp_poses) == 0:
                continue

            #print("REF TIME", reference_seconds, "COMP TIME", comparison_seconds, "REF FRAME", reference_frame, "COMP FRAME", comparison_frame)
            #print("# REF POSES:", len(matching_ref_poses), "# COMP POSES", len(matching_comp_poses))

            # Should this be on the reference video's timeline, or the comparison's?
            # Probably reference, if we're using it as the "yardstick" for all of the other videos.
            sim_times.append(reference_seconds)

            # Keep track of matching poses to generate global "consensus" poses
            if reference_seconds not in aligned_ref_poses:
                # XXX List is probably not necessary here
                aligned_ref_poses[reference_seconds] = [matching_ref_poses]
                aligned_ref_frames[reference_seconds] = [matching_ref_frames]
            if reference_seconds not in aligned_comp_poses:
                aligned_comp_poses[reference_seconds] = [matching_comp_poses]
                aligned_comp_frames[reference_seconds] = [matching_comp_frames]
            else:
                aligned_comp_poses[reference_seconds].append(matching_comp_poses)
                aligned_comp_frames[reference_seconds].append(matching_comp_frames)

            # Should just compute cosine similarity for all poses in both matched frames
            ref_poems = np.stack(matching_ref_poses["poem_embedding"].to_list(), axis=0)
            comp_poems = np.stack(matching_comp_poses["poem_embedding"].to_list(), axis=0)
            poem_sims = cosine_similarity(ref_poems, comp_poems)
            avg_poem_sim = np.mean(poem_sims.flatten())
            poem_sim_values.append(avg_poem_sim)

            ref_global3ds = np.stack(matching_ref_poses["global3d_coco13"].values, axis=0)
            comp_global3ds = np.stack(matching_comp_poses["global3d_coco13"].values, axis=0)
            global3d_sims = cosine_similarity(ref_global3ds, comp_global3ds)
            avg_global3d_sim = np.mean(global3d_sims.flatten())
            global3d_sim_values.append(avg_global3d_sim)

            ref_avas = np.stack(matching_ref_poses["ava_action"].values, axis=0)
            comp_avas = np.stack(matching_comp_poses["ava_action"].values, axis=0)
            ava_sims = cosine_similarity(ref_avas, comp_avas)
            avg_ava_sim = np.mean(ava_sims.flatten())
            ava_sim_values.append(avg_ava_sim)

            ref_movement3d = matching_ref_frames["movement3d"].values[0]
            comp_movement3d = matching_comp_frames["movement3d"].values[0]
            #movement3d_sims = cosine_similarity([ref_movement3ds], [comp_movement3ds])
            #avg_movement3d_sim = np.mean(movement3d_sims.flatten())
            movement3d_sim_values.append(comp_movement3d - ref_movement3d)
            raw_comp_movement3d.append(comp_movement3d)
            raw_ref_movement3d.append(ref_movement3d)

    # XXX Consider smoothing the time series
    # Maybe also plot all 3 on the same graph?

    poem_sim_values = smooth_series(poem_sim_values)
    global3d_sim_values = smooth_series(global3d_sim_values)
    ava_sim_values = smooth_series(ava_sim_values)

    fig = plt.figure(figsize=(10,4))
    plt.title(f"{comparison_video} - POEM")
    plt.plot(sim_times, poem_sim_values)
    for boundary in REF_BOUNDARIES:
        plt.axvline(x=boundary, color='r', linestyle="--")
    for boundary in REF_ACTS:
        plt.axvline(x=boundary, color='orange', linestyle="--")
    ax = plt.gca()
    ax.xaxis.set_major_formatter(plt.FuncFormatter(HMS))
    plt.savefig(f"stylometry/alignment_files_4096/{comparison_video.replace('.mp4', '_poem.png')}")
    plt.close()
    print("POEM data points:", len(sim_times))

    fig = plt.figure(figsize=(10,4))
    plt.title(f"{comparison_video} - 3D poses")
    plt.plot(sim_times, global3d_sim_values)
    for boundary in REF_BOUNDARIES:
        plt.axvline(x=boundary, color='r', linestyle="--")
    for boundary in REF_ACTS:
        plt.axvline(x=boundary, color='orange', linestyle="--")
    ax = plt.gca()
    ax.xaxis.set_major_formatter(plt.FuncFormatter(HMS))
    plt.savefig(f"stylometry/alignment_files_4096/{comparison_video.replace('.mp4', '_global3d.png')}")
    plt.close()
    print("Global 3D data points:", len(sim_times))

    fig = plt.figure(figsize=(10,4))
    plt.title(f"{comparison_video} - actions")
    ax = plt.gca()
    plt.plot(sim_times, ava_sim_values)
    for boundary in REF_BOUNDARIES:
        plt.axvline(x=boundary, color='r', linestyle="--")
    for boundary in REF_ACTS:
        plt.axvline(x=boundary, color='orange', linestyle="--")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(HMS))
    plt.savefig(f"stylometry/alignment_files_4096/{comparison_video.replace('.mp4', '_ava.png')}")
    plt.close()
    print("AVA data points:", len(sim_times))

    fig = plt.figure(figsize=(10,4))
    plt.title(f"{comparison_video} - 3D movement")
    ax = plt.gca()
    plt.plot(sim_times, movement3d_sim_values)
    for boundary in REF_BOUNDARIES:
        plt.axvline(x=boundary, color='r', linestyle="--")
    for boundary in REF_ACTS:
        plt.axvline(x=boundary, color='orange', linestyle="--")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(HMS))
    plt.savefig(f"stylometry/alignment_files_4096/{comparison_video.replace('.mp4', '_movement3d.png')}")
    plt.close()
    print("3D movement data points:", len(sim_times))

    if not plotted_ref_movement:
        ax = all_movement3d_fig.gca()
        ax.plot(sim_times, raw_ref_movement3d, color=video_colors[0], linestyle=":", alpha=0.2)
        plotted_ref_movement = True

    ax = all_movement3d_fig.gca()
    ax.plot(sim_times, raw_comp_movement3d, color=video_color, linestyle=":", alpha=0.2)

ax = all_movement3d_fig.gca()
for boundary in REF_BOUNDARIES:
    ax.axvline(x=boundary, color='r', linestyle="--")
for boundary in REF_ACTS:
    ax.axvline(x=boundary, color='orange', linestyle="--")
all_movement3d_fig.legend(video_aliases_legend, handler_map={PathCollection : HandlerPathCollection(update_func=update_legend), plt.Line2D : HandlerLine2D(update_func=update_legend)}, loc="lower right")
all_movement3d_fig.savefig(f"stylometry/alignment_files_4096/all_movement3d.png")

In [ ]:
# Determine consensus poses, embeddings
poem_consensus = []
global3d_consensus = []
ava_consensus = []
movement3d_consensus = []

aligned_times = sorted(list(aligned_ref_poses.keys()))

# Because we only consider aligned frames/timecodes with at least one set of reference AND comparison
# video poses, both sets (reference and comparsion) should have the same timecodes
for sim_time in aligned_times:
    poem_consensus.append(np.mean(np.vstack([np.stack(aligned_ref_poses[sim_time][0]["poem_embedding"]), np.stack([apose["poem_embedding"].values[0] for apose in aligned_comp_poses[sim_time]])]), axis=0))
    global3d_consensus.append(np.mean(np.vstack([np.stack(aligned_ref_poses[sim_time][0]["global3d_coco13"]), np.stack([apose["global3d_coco13"].values[0] for apose in aligned_comp_poses[sim_time]])]), axis=0))
    ava_consensus.append(np.mean(np.vstack([np.stack(aligned_ref_poses[sim_time][0]["ava_action"]), np.stack([apose["ava_action"].values[0] for apose in aligned_comp_poses[sim_time]])]), axis=0))
    all_frame_movement3ds = [aligned_ref_frames[sim_time][0]["movement3d"].values[0]] + [apose["movement3d"].values[0] for apose in aligned_comp_frames[sim_time]]
    movement3d_consensus.append(np.mean(all_frame_movement3ds))

# The consensus poses and embeddings become the "reference" performance (we can also compare the original "reference" performance to it)

print("Shape of poem consensus", np.array(poem_consensus).shape)
print("Shape of global 3D consensus", np.array(global3d_consensus).shape)
print("Shape of AVA consensus", np.array(ava_consensus).shape)
print("Shape of movement 3D consensus", np.array(movement3d_consensus).shape)

print("length of sim times", len(sim_times))

In [ ]:
from matplotlib.collections import PathCollection
from matplotlib.legend_handler import HandlerPathCollection, HandlerLine2D

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool
from bokeh.models import ColumnDataSource
output_notebook()

all_xs = []
poem_ys = []
global3d_ys = []
ava_ys = []
movement3d_ys = []

video_aliases_legend = [video_aliases[video_name] for video_name in all_videos]

def update_legend(handle, orig):
    handle.update_from(orig)
    handle.set_alpha(1)

poem_fig = plt.figure(figsize=(16,6))
ax = poem_fig.gca()
ax.set_title("POEM embeddings vs. consensus")
ax.xaxis.set_major_formatter(plt.FuncFormatter(HMS))

global3d_fig = plt.figure(figsize=(16,6))
ax = global3d_fig.gca()
ax.set_title("3D poses vs. consensus")
ax.xaxis.set_major_formatter(plt.FuncFormatter(HMS))

ava_fig = plt.figure(figsize=(16,6))
ax = ava_fig.gca()
ax.set_title("Action embeddings vs. consensus")
ax.xaxis.set_major_formatter(plt.FuncFormatter(HMS))

movement3d_fig = plt.figure(figsize=(16,6))
ax = movement3d_fig.gca()
ax.set_title("3D movement vs. consensus")
ax.xaxis.set_major_formatter(plt.FuncFormatter(HMS))

stats_file = open("stylometry/alignment_files_4096/consensus_comparisons.tsv", "w", encoding="utf-8")

stats_file.write("comparison\tpoem_mean\tpoem_median\tpoem_stdev\t3d_mean\t3d_median\t3d_stdev\tava_mean\tava_median\tava_stdev\n")

for c, comparison_video in enumerate(all_videos):

    sim_times = []
    poem_sim_values = []
    global3d_sim_values = []
    ava_sim_values = []
    movement3d_sim_values = []

    if comparison_video != reference_video:

        print("Getting comparison data for video", comparison_video)
        comparison_id, comparison_fps = get_id_and_fps(comparison_video, videos_df)
        comparison_poses = await db.get_pose_data_from_video(comparison_id)
        comparison_frames = await db.get_pose_data_by_frame(comparison_id)
        comp_poses_df = pd.DataFrame.from_records(comparison_poses, columns=comparison_poses[0].keys())
        comp_frames_df = pd.DataFrame.from_records(comparison_frames, columns=comparison_frames[0].keys())

        with open(f"stylometry/alignment_files_4096/{comparison_video}_warping.txt", "r", encoding="utf-8") as warping_data:
            for line in warping_data:
                line_data = line.strip().split("\t")
                if line_data[0] == 0 or line_data[3].endswith("*"):
                    continue
                reference_seconds = float(line_data[2])
                comparison_seconds = float(line_data[3])

                if reference_seconds not in aligned_times:
                    continue

                comparison_frame = round(comparison_seconds * comparison_fps)
                matching_comp_poses = comp_poses_df[comp_poses_df["frame"] == comparison_frame]
                matching_comp_frame = comp_frames_df[comp_frames_df["frame"] == comparison_frame]

                if len(matching_comp_poses) == 0:
                    continue

                consensus_index = aligned_times.index(reference_seconds)
                sim_times.append(reference_seconds)

                # Should just compute cosine similarity for all poses in both matched frames
                consensus_poem = poem_consensus[consensus_index]
                consensus_global3d = global3d_consensus[consensus_index]
                consensus_ava = ava_consensus[consensus_index]
                consensus_movement3d = movement3d_consensus[consensus_index]
                
                comp_poem = np.mean(np.stack(matching_comp_poses["poem_embedding"].to_list(), axis=0), axis=0)
                poem_sim = cosine_similarity([consensus_poem], [comp_poem])
                poem_sim_values.append(poem_sim[0])

                comp_global3d = np.mean(np.stack(matching_comp_poses["global3d_coco13"].values, axis=0), axis=0)
                global3d_sim = cosine_similarity([consensus_global3d], [comp_global3d])
                global3d_sim_values.append(global3d_sim[0])

                comp_ava = np.mean(np.stack(matching_comp_poses["ava_action"].values, axis=0), axis=0)
                ava_sim = cosine_similarity([consensus_ava], [comp_ava])
                ava_sim_values.append(ava_sim[0])

                ref_movement3d = matching_ref_frames["movement3d"].values[0]
                comp_movement3d = matching_comp_frame["movement3d"].values[0]
                movement3d_sim_values.append(comp_movement3d - consensus_movement3d)

    else:

        for consensus_seconds in aligned_times:
            consensus_index = aligned_times.index(consensus_seconds)
            reference_frame = round(consensus_seconds * reference_fps)

            matching_ref_poses = ref_poses_df[ref_poses_df["frame"] == reference_frame]
            matching_ref_frames = ref_frames_df[ref_frames_df["frame"] == reference_frame]

            if len(matching_ref_poses) == 0:
                continue

            sim_times.append(consensus_seconds)

            # Should just compute cosine similarity for all poses in both matched frames
            consensus_poem = poem_consensus[consensus_index]
            consensus_global3d = global3d_consensus[consensus_index]
            consensus_ava = ava_consensus[consensus_index]
            consensus_movement3d = movement3d_consensus[consensus_index]
            
            ref_poem = np.mean(np.stack(matching_ref_poses["poem_embedding"].to_list(), axis=0), axis=0)
            poem_sim = cosine_similarity([consensus_poem], [ref_poem])
            poem_sim_values.append(poem_sim[0])

            ref_global3d = np.mean(np.stack(matching_ref_poses["global3d_coco13"].values, axis=0), axis=0)
            global3d_sim = cosine_similarity([consensus_global3d], [ref_global3d])
            global3d_sim_values.append(global3d_sim[0])

            ref_ava = np.mean(np.stack(matching_ref_poses["ava_action"].values, axis=0), axis=0)
            ava_sim = cosine_similarity([consensus_ava], [ref_ava])
            ava_sim_values.append(ava_sim[0])

            ref_movement3d = matching_ref_frames["movement3d"].values[0]
            movement3d_sim = ref_movement3d - consensus_movement3d
            movement3d_sim_values.append(movement3d_sim)

    # XXX Consider smoothing the time series
    # Maybe also plot all 3 on the same graph?

    print(f"{comparison_video} vs. consensus, total times {len(sim_times)}, length of POEM values {len(poem_sim_values)}")

    poem_sim_values = smooth_series(poem_sim_values, 300)
    global3d_sim_values = smooth_series(global3d_sim_values, 300)
    ava_sim_values = smooth_series(ava_sim_values, 300)

    all_xs.append(sim_times)
    poem_ys.append(poem_sim_values)
    global3d_ys.append(global3d_sim_values)
    ava_ys.append(ava_sim_values)
    movement3d_ys.append(movement3d_sim_values)

    video_color = video_colors[c]

    ax = poem_fig.gca()
    ax.plot(sim_times, poem_sim_values, color=video_color, linestyle=":", alpha=0.2)

    poem_stats = get_distribution_stats(poem_sim_values, False)
    print("POEM:", poem_stats)

    ax = global3d_fig.gca()
    ax.plot(sim_times, global3d_sim_values, color=video_color, linestyle=":", alpha=0.2)

    global3d_stats = get_distribution_stats(global3d_sim_values, False)
    print("3D POSE:", global3d_stats)

    ax = ava_fig.gca()
    ax.plot(sim_times, ava_sim_values, color=video_color, linestyle=":", alpha=0.2)

    ava_stats = get_distribution_stats(ava_sim_values, False)
    print("ACTIONS:", ava_stats)

    ax = movement3d_fig.gca()
    ax.plot(sim_times, movement3d_sim_values, color=video_color, linestyle=":", alpha=0.2)

    comparison_stats = [comparison_video, poem_stats["mean"], poem_stats["median"], poem_stats["stdev"], global3d_stats["mean"], global3d_stats["median"], global3d_stats["stdev"], ava_stats["mean"], ava_stats["median"], ava_stats["stdev"]]
    stats_file.write("\t".join([str(stat) for stat in comparison_stats]) + "\n")

ax = poem_fig.gca()
for boundary in REF_BOUNDARIES:
    ax.axvline(x=boundary, color='r', linestyle="--")
for boundary in REF_ACTS:
    ax.axvline(x=boundary, color='orange', linestyle="--")
poem_fig.legend(video_aliases_legend, handler_map={PathCollection : HandlerPathCollection(update_func=update_legend), plt.Line2D : HandlerLine2D(update_func=update_legend)}, loc="lower right")
poem_fig.savefig(f"stylometry/alignment_files_4096/consensus_all_poem.png")
#plt.close(poem_fig)

ax = global3d_fig.gca()
for boundary in REF_BOUNDARIES:
    ax.axvline(x=boundary, color='r', linestyle="--")
for boundary in REF_ACTS:
    ax.axvline(x=boundary, color='orange', linestyle="--")
global3d_fig.legend(video_aliases_legend, handler_map={PathCollection : HandlerPathCollection(update_func=update_legend), plt.Line2D : HandlerLine2D(update_func=update_legend)}, loc="lower right")
global3d_fig.savefig(f"stylometry/alignment_files_4096/consensus_all_global3d.png")
#plt.close(global3d_fig)

ax = ava_fig.gca()
for boundary in REF_BOUNDARIES:
    ax.axvline(x=boundary, color='r', linestyle="--")
for boundary in REF_ACTS:
    ax.axvline(x=boundary, color='orange', linestyle="--")
ava_fig.legend(video_aliases_legend, handler_map={PathCollection : HandlerPathCollection(update_func=update_legend), plt.Line2D : HandlerLine2D(update_func=update_legend)}, loc="lower right")
ava_fig.savefig(f"stylometry/alignment_files_4096/consensus_all_ava.png")
#plt.close(ava_fig)

ax = movement3d_fig.gca()
for boundary in REF_BOUNDARIES:
    ax.axvline(x=boundary, color='r', linestyle="--")
for boundary in REF_ACTS:
    ax.axvline(x=boundary, color='orange', linestyle="--")
movement3d_fig.legend(video_aliases_legend, handler_map={PathCollection : HandlerPathCollection(update_func=update_legend), plt.Line2D : HandlerLine2D(update_func=update_legend)}, loc="lower right")
movement3d_fig.savefig(f"stylometry/alignment_files_4096/consensus_all_movement3d.png")
#plt.close(ava_fig)

stats_file.close()

In [ ]:
from bokeh.plotting import figure, output_file, save
from bokeh.models import DatetimeTickFormatter, SingleIntervalTicker

# set output to static HTML file
output_file(filename=f"stylometry/alignment_files_4096/dg_poem_comparison.html", title="Don Giovanni pose embedding comparison")

p = figure(width=1200, height=600, x_axis_type="datetime")

p.title.text = "Similarity of each performance's pose embeddings to the global consensus"

p.xaxis.formatter = DatetimeTickFormatter(hours="%H:%M:%S",
                                          hourmin="%H:%M:%S",
                                          minutes="%H:%M:%S",
                                          minsec="%H:%M:%S",
                                          seconds="%H:%M:%S")

p.xaxis.ticker = SingleIntervalTicker(interval=1200000)

min_y = 0
max_y = 0

for v, video_name in enumerate(all_videos):

    video_alias = video_aliases[video_name]

    dts = pd.DataFrame({"time": all_xs[v]})
    dts["time"] = pd.to_datetime(dts["time"], unit='s', errors='coerce')

    ys = np.array(poem_ys[v]).flatten()

    min_y = min(np.min(ys), min_y)
    max_y = max(np.max(ys), max_y)

    p.line(dts["time"], ys, line_width=2, color=video_colors[v], alpha=0.5, legend_label=video_alias)

for boundary in REF_BOUNDARIES:
    boundary_dts = pd.DataFrame({"time": [boundary]})
    boundary_dts["time"] = pd.to_datetime(boundary_dts["time"], unit='s', errors='coerce')
    p.line([boundary_dts["time"], boundary_dts["time"]], [min_y, max_y], color='red', line_dash="dashed", line_width=2)
for boundary in REF_ACTS:
    boundary_dts = pd.DataFrame({"time": [boundary]})
    boundary_dts["time"] = pd.to_datetime(boundary_dts["time"], unit='s', errors='coerce')
    p.line([boundary_dts["time"], boundary_dts["time"]], [min_y, max_y], color='red', line_dash="dashed", line_width=2)

p.legend.location = "bottom_right"
p.legend.click_policy="hide"

save(p)
#show(p)

